### Shared Imports

In [1]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV,RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (mean_squared_error, r2_score, mean_absolute_error,
                             classification_report, confusion_matrix, accuracy_score,
                             silhouette_score, davies_bouldin_score)

#Regression
from sklearn.ensemble import AdaBoostRegressor 

#Classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier

#Clustering
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')

##### Dataset Loaded

In [2]:
df = pd.read_csv(r'X:\nasim_xhqpjmy\Code\MLops\Race-Telemetry\dataset\data.csv')
df.head()

,Unnamed: 0,since_last_ns,timestamp_ms,current_engine_rpm,wheel_rotation_speed_front_left,wheel_rotation_speed_front_right,wheel_rotation_speed_rear_left,wheel_rotation_speed_rear_right,wheel_on_rumble_strip_front_left,wheel_on_rumble_strip_front_right,...,clutch,handbrake,gear,steer,lap_number,best_lap_time,last_lap_time,current_lap_time,current_race_time,race_position
0,0,0,27901812,4985.9053,66.646460,66.550380,65.249550,65.155790,False,False,...,0,0,2,1,0,0.0,0.0,0.000000,0.000000,1
1,1,7438900,27901828,4985.5083,66.634400,66.541245,65.254670,65.164270,False,False,...,0,0,2,1,0,0.0,0.0,0.008331,0.008331,1
2,2,8171000,27901828,4964.7420,66.630806,66.537970,65.239240,65.138350,False,False,...,0,0,2,1,0,0.0,0.0,0.016670,0.016670,1
3,3,8336100,27901843,4949.4760,66.613190,66.528110,65.206604,65.110146,False,False,...,0,0,2,1,0,0.0,0.0,0.025003,0.025003,1
4,4,8430500,27901843,4939.9850,66.604416,66.525300,65.134160,65.053990,False,False,...,0,0,2,1,0,0.0,0.0,0.033348,0.033348,1


##### Handling Null Values

In [3]:
df.isnull().sum()

Unnamed: 0                         0
since_last_ns                      0
timestamp_ms                       0
current_engine_rpm                 0
wheel_rotation_speed_front_left    0
                                  ..
best_lap_time                      0
last_lap_time                      0
current_lap_time                   0
current_race_time                  0
race_position                      0
Length: 73, dtype: int64

## Shared Feature Engineering

In [4]:
# Create a copy for processing
df_processed = df.copy()

df_processed = df_processed.fillna(0)

# Wheel slip magnitude (combined front and rear)
df_processed['wheel_slip_magnitude_front'] = np.sqrt(
    df_processed['tire_slip_rotation_front_left']**2 + 
    df_processed['tire_slip_rotation_front_right']**2
)
df_processed['wheel_slip_magnitude_rear'] = np.sqrt(
    df_processed['tire_slip_rotation_rear_left']**2 + 
    df_processed['tire_slip_rotation_rear_right']**2
)

# Tire stress (combined slip)
df_processed['tire_stress_front'] = (
    df_processed['tire_combined_slip_front_left'] + 
    df_processed['tire_combined_slip_front_right']
) / 2
df_processed['tire_stress_rear'] = (
    df_processed['tire_combined_slip_rear_left'] + 
    df_processed['tire_combined_slip_rear_right']
) / 2

# Average tire temperature
df_processed['avg_tire_temp'] = (
    df_processed['tire_temp_front_left'] + 
    df_processed['tire_temp_front_right'] +
    df_processed['tire_temp_rear_left'] + 
    df_processed['tire_temp_rear_right']
) / 4

# Suspension travel metrics
df_processed['avg_suspension_travel'] = (
    df_processed['suspension_travel_meters_front_left'] +
    df_processed['suspension_travel_meters_front_right'] +
    df_processed['suspension_travel_meters_rear_left'] +
    df_processed['suspension_travel_meters_rear_right']
) / 4

# Acceleration magnitude
df_processed['acceleration_magnitude'] = np.sqrt(
    df_processed['acceleration_x']**2 + 
    df_processed['acceleration_y']**2 + 
    df_processed['acceleration_z']**2
)

# Velocity magnitude
df_processed['velocity_magnitude'] = np.sqrt(
    df_processed['velocity_x']**2 + 
    df_processed['velocity_y']**2 + 
    df_processed['velocity_z']**2
)

# Steering rate (change in steering)
df_processed['steering_rate'] = df_processed['steer'].diff().fillna(0)

# Brake-acceleration interaction
df_processed['brake_accel_interaction'] = df_processed['brake'] * df_processed['acceleration']

# RPM per speed ratio
df_processed['rpm_speed_ratio'] = df_processed['current_engine_rpm'] / (df_processed['speed'] + 1)

print(f"Total features after engineering: {df_processed.shape[1]}")

Total features after engineering: 84


## Shared Data preprocessing

In [ ]:
#Feature selection
numeric_features = [
    'current_engine_rpm', 'speed', 'power', 'torque', 'boost', 'fuel',
    'acceleration_x', 'acceleration_y', 'acceleration_z',
    'velocity_x', 'velocity_y', 'velocity_z',
    'angular_velocity_x', 'angular_velocity_y', 'angular_velocity_z',
    'yaw', 'pitch', 'roll',
    'wheel_slip_magnitude_front', 'wheel_slip_magnitude_rear',
    'tire_stress_front', 'tire_stress_rear',
    'avg_tire_temp', 'avg_suspension_travel',
    'acceleration_magnitude', 'velocity_magnitude',
    'steering_rate', 'rpm_speed_ratio',
    'steer', 'distance_traveled'
]
categorical_features = ['gear', 'lap_number', 'race_position']

# Encode categorical features
le_gear = LabelEncoder()
le_lap = LabelEncoder()
le_pos = LabelEncoder()

df_processed['gear_encoded'] = le_gear.fit_transform(df_processed['gear'].astype(str))
df_processed['lap_encoded'] = le_lap.fit_transform(df_processed['lap_number'].astype(str))
df_processed['position_encoded'] = le_pos.fit_transform(df_processed['race_position'].astype(str))

#Feature concatenation
all_features = numeric_features + ['gear_encoded', 'lap_encoded', 'position_encoded']

#Normalization
print("\nNormalizing features...")
scaler = StandardScaler()
df_processed[numeric_features] = scaler.fit_transform(df_processed[numeric_features])
print("Preprocessing complete!")


Normalizing features...
Preprocessing complete!


### Training Regression Model -Lap time Prediction

In [6]:
lap_time_features = [
    'speed', 'current_engine_rpm', 'acceleration_magnitude',
    'velocity_magnitude', 'tire_stress_front', 'tire_stress_rear',
    'wheel_slip_magnitude_front', 'wheel_slip_magnitude_rear',
    'avg_tire_temp', 'power', 'torque', 'boost',
    'position_x', 'position_y', 'position_z',
    'yaw', 'pitch', 'roll', 'gear_encoded', 'steer'
]

# Prepare data
X_lap = df_processed[lap_time_features]
y_lap = df_processed['current_lap_time']

# Remove rows where lap time is 0 or invalid
valid_idx = y_lap > 0
X_lap = X_lap[valid_idx]
y_lap = y_lap[valid_idx]

X_lap_train, X_lap_test, y_lap_train, y_lap_test = train_test_split(X_lap, y_lap, test_size=0.2, random_state=42)

params = {
    'n_estimators': 100,
    'learning_rate': 0.5,
    'loss': 'square'
}

best_model = AdaBoostRegressor(**params, random_state=42).fit(X_lap_train, y_lap_train)

# Evaluate
def metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return rmse, mae, r2

train_rmse, train_mae, train_r2 = metrics(y_lap_train, best_model.predict(X_lap_train))
test_rmse, test_mae, test_r2 = metrics(y_lap_test, best_model.predict(X_lap_test))

print(f"Train RMSE: {train_rmse:.5f} | Test RMSE: {test_rmse:.5f}")
print(f"Train MAE: {train_mae:.5f} | Test MAE: {test_mae:.5f}")
print(f"Train R²: {train_r2:.5f} | Test R²: {test_r2:.5f}")

save_dir = "final_models"
os.makedirs(save_dir, exist_ok=True)  

model_path = os.path.join(save_dir, "Regression.pkl")
joblib.dump(best_model, model_path)

print(f"Model saved successfully at {model_path}")

Train RMSE: 14.54328 | Test RMSE: 14.59903
Train MAE: 11.50236 | Test MAE: 11.49131
Train R²: 0.89800 | Test R²: 0.89680
Model saved successfully at final_models\Regression.pkl


### Training Classification Model - Gear,Brake Prediction

In [7]:
gear_brake_features = [
    'current_engine_rpm', 'speed', 'acceleration_magnitude',
    'velocity_magnitude', 'power', 'torque', 'rpm_speed_ratio',
    'acceleration_x', 'acceleration_y', 'acceleration_z',
    'brake_accel_interaction', 'steer', 'steering_rate'
]

X_gear_brake = df_processed[gear_brake_features]
# Prepare targets: gear and brake (binary)
df_processed['brake_binary'] = (df_processed['brake'] > 0).astype(int)
y_gear = df_processed['gear']
y_brake = df_processed['brake_binary']

y_gear_brake = pd.DataFrame({
    'gear': y_gear,
    'brake': y_brake
})

X_gb_train, X_gb_test, y_gb_train, y_gb_test = train_test_split(
    X_gear_brake, y_gear_brake, test_size=0.2, random_state=42
)

best_params = {
    'n_estimators': 10,
    'max_depth': 5,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'max_features': 'sqrt'
}

best_rf = MultiOutputClassifier(RandomForestClassifier(random_state=42, n_jobs=-1, **best_params))

print("\nTraining RandomForest Classifier with best parameters...")
best_rf.fit(X_gb_train, y_gb_train)

y_pred_train = best_rf.predict(X_gb_train)
y_pred_test = best_rf.predict(X_gb_test)

y_pred_train_gear, y_pred_train_brake = y_pred_train[:, 0], y_pred_train[:, 1]
y_pred_test_gear, y_pred_test_brake = y_pred_test[:, 0], y_pred_test[:, 1]

#Evaluation 
train_acc_gear = accuracy_score(y_gb_train['gear'], y_pred_train_gear)
test_acc_gear = accuracy_score(y_gb_test['gear'], y_pred_test_gear)

train_acc_brake = accuracy_score(y_gb_train['brake'], y_pred_train_brake)
test_acc_brake = accuracy_score(y_gb_test['brake'], y_pred_test_brake)

print("\nTest Accuracies:")
print(f" Gear:  {test_acc_gear:.4f}")
print(f" Brake: {test_acc_brake:.4f}")
print("\nClassification Report (Gear):")
print(classification_report(y_gb_test['gear'], y_pred_test_gear))
print("\nClassification Report (Brake):")
print(classification_report(y_gb_test['brake'], y_pred_test_brake))
print("\nTraining completed successfully.")

save_dir = "final_models"
os.makedirs(save_dir, exist_ok=True)  

model_path = os.path.join(save_dir, "Classification.pkl")
joblib.dump(best_model, model_path)

print(f"Model saved successfully at {model_path}")


Training RandomForest Classifier with best parameters...

Test Accuracies:
 Gear:  0.9860
 Brake: 0.9775

Classification Report (Gear):
              precision    recall  f1-score   support

           1       0.99      0.96      0.97       408
           2       0.99      0.96      0.97      1584
           3       0.99      0.98      0.99      8334
           4       0.98      1.00      0.99      8578

    accuracy                           0.99     18904
   macro avg       0.99      0.97      0.98     18904
weighted avg       0.99      0.99      0.99     18904


Classification Report (Brake):
              precision    recall  f1-score   support

           0       0.99      0.99      0.99     15868
           1       0.93      0.93      0.93      3036

    accuracy                           0.98     18904
   macro avg       0.96      0.96      0.96     18904
weighted avg       0.98      0.98      0.98     18904


Training completed successfully.
Model saved successfully at final_m

### Training Clustering Model - Driving Behavious analysis

In [8]:
clustering_features = [
    'tire_slip_angle_front_left', 'tire_slip_angle_front_right',
    'tire_slip_angle_rear_left', 'tire_slip_angle_rear_right',
    'wheel_slip_magnitude_front', 'wheel_slip_magnitude_rear',
    'tire_stress_front', 'tire_stress_rear',
    'acceleration_magnitude', 'brake', 'acceleration',
    'avg_suspension_travel', 'steer', 'steering_rate',
    'angular_velocity_x', 'angular_velocity_y', 'angular_velocity_z'
]

X_cluster = df_processed[clustering_features]

if len(X_cluster) > 50000:
    sample_indices = np.random.choice(len(X_cluster), 50000, replace=False)
    X_cluster_sample = X_cluster.iloc[sample_indices]
else:
    X_cluster_sample = X_cluster

pca_params = {
    "n_components": 2,
    "random_state": 42
}

kmeans_params = {
    "n_clusters": 3,
    "init": "k-means++",
    "max_iter": 20,
    "batch_size": 512,
    "tol": 1e-2,
    "random_state": 42
}

pca = PCA(**pca_params)
X_reduced = pca.fit_transform(X_cluster_sample)

kmeans = MiniBatchKMeans(**kmeans_params)
kmeans.fit(X_reduced)

inertia = kmeans.inertia_
silhouette = silhouette_score(X_reduced, kmeans.labels_)

print(f"Inertia: {inertia:.2f}")
print(f"Silhouette Score: {silhouette:.3f}")


save_dir = "final_models"
os.makedirs(save_dir, exist_ok=True)  

model_path = os.path.join(save_dir, "Clustering.pkl")
joblib.dump(best_model, model_path)

print(f"Model saved successfully at {model_path}")

Inertia: 77674489.53
Silhouette Score: 0.768
Model saved successfully at final_models\Clustering.pkl
